# Import Libraries

In [1]:
import pandas as pd
import json
import numpy as np

from textwrap import dedent
from typing import Literal, Mapping, TypedDict
import ast

In [2]:
import boto3

# Boto3 Setup

In [3]:
#Check the latest version of boto3

print("boto3 version: ", boto3.__version__)

print("pandas version: ", pd.__version__)
import json
print('json version: ', json.__version__)
from botocore.exceptions import ClientError

region = "us-east-1"

session = boto3.Session(profile_name="AWS_bedrock_NHL")

#Create Bedrock and Bedrock runtime clients
bedrock = session.client("bedrock", region_name=region)
br = session.client("bedrock-runtime", region_name=region)

boto3 version:  1.43.68
pandas version:  2.3.3
json version:  2.0.9


# Config

In [4]:
model_id_name_dict  = {'ug3pv3tm0cqj':'amazon-nova-pro-v1-0',
                     'jclyy0txowgp':'mistral-7b-instruct-v0-2',
                      'jfj5rgftsdq5':'meta-llama3-1-70b-instruct-v1-0',
                      'vjs5dyljcm0y':'meta-llama3-1-8b-instruct-v1-0',
                      'uyrecy708aqn':'mixtral-8x7b-instruct-v0-1',
                      'gaawkqvz1uc3': 'anthropic-claude-opus-4-6-v1',
                      'z1q2q8peq81j' : 'claude-haiku-4-5-v1-0',
                      'RB_App':'Rule-Based_Algorithm'}
class syn_config:
    model_id = 'gaawkqvz1uc3'
    model_name = model_id_name_dict[model_id]
    n_generate_single = 5
    n_generate_mixture = 5

    temperature = 0.3 

    add_evidence = True


    child_family_care = '''**child_family_care**
This domain covers child care, daycare, dependent care, or difficulty paying for child care. 
Positive examples: needs daycare, cannot afford child care, child care is a barrier. 
Negative examples: no child care needs, child care is adequate, dependent care needs are met. 
Unclear examples: has children, child care discussed, family responsibilities mentioned without stating a need. 
'''
    
    education = ''' **education** 
This domain covers educational attainment or education-related disadvantage. 
Positive examples: less than high school diploma, very limited schooling, education level suggests disadvantage. 
Negative examples: high school graduate, GED, or higher education clearly documented. 
Unclear examples: education discussed without a clear level, school history mentioned without enough detail. 
Important: do not treat any mention of school as positive unless low attainment or education-related disadvantage is clearly documented.  
'''

    employment = '''**employment** 
This domain covers unemployment, job instability, or needing help finding work. 
Positive examples: unemployed and looking for work, wants help finding a job, cannot keep a job. 
Negative examples: employed, stable work, no employment assistance needed. 
Unclear examples: job discussed without clear current status, retired/disability status mentioned without clear employment need. 
    '''
    financial_strain = '''**financial_strain**
This domain covers difficulty paying for basics such as food, housing, heating, medical care, medications, or other essential expenses. 
Positive examples: somewhat hard or very hard to pay for basics, trouble paying for medicines, cannot afford bills. 
Negative examples: not hard to pay for basics, denies financial concerns, medications are affordable. 
Unclear examples: money concerns mentioned without indicating whether basic needs are affected. 
'''

    food_insecurity = '''**food_insecurity**
This domain covers worrying food will run out, food actually running out, skipping meals, or not having enough money for food. 
Positive examples: often true/sometimes true that food runs out, household ran out of food, cannot afford food. 
Negative examples: food secure, denies food insecurity, enough food available. 
Unclear examples: appetite issues, diet changes, or nutrition counseling without evidence of access problems. 
    '''

    housing_instability = '''**housing_instability**
This domain covers unstable housing, homelessness risk, risk of losing housing, eviction risk, or inability to pay rent/mortgage on time. 
Positive examples: worried about losing housing, no steady place to live, unable to pay rent or mortgage on time. 
Negative examples: steady place to live, housing stable, denies housing concerns. 
Unclear examples: housing mentioned without stability status, temporarily staying somewhere but unclear whether this reflects instability. 
'''

    housing_quality = '''**housing_quality**
This domain covers unsafe or poor housing conditions. 
Positive examples: pests, bugs, mice, inadequate heat, mold, leaks, unsafe housing conditions. 
Negative examples: home conditions are adequate, safe, or no housing quality problems. 
Unclear examples: housing discussed without condition details.

    '''

    relationship_safety = '''**relationship_safety**
This domain covers threats, physical harm, abuse, or feeling unsafe with family, friends, partner, or others. 
Positive examples: threatened with harm, physically hurt, abuse, feels unsafe in relationship/home. 
Negative examples: denies abuse, denies violence, feels safe. 
Unclear examples: safety screening mentioned without result, interpersonal conflict mentioned without clear threat/harm. 

    '''

    social_isolation = '''**social_isolation**
This domain covers loneliness, isolation, or very limited meaningful contact with others. 
Positive examples: feels lonely, feels isolated, sees/talks to close people less than once a week. 
Negative examples: regular supportive contact, denies loneliness or isolation. 
Unclear examples: lives alone, few family details, limited social history without clear loneliness or support deficit. 
'''

    stress = '''**stress** 
This domain covers clinically meaningful current stress burden. 
Positive examples: stressed, quite a bit of stress, ongoing stress affecting daily life. 
Negative examples: denies stress, no current stress concerns. 
Unclear examples: anxiety/depression history without explicit current stress, stress discussed in general terms only. 
Important: do not classify a psychiatric diagnosis alone as stress unless the note explicitly describes current stress. 
'''

    transportation = '''**transportation**
This domain covers lack of transportation interfering with medical appointments, medications, work, meetings, or daily living needs. 
Positive examples: transportation kept patient from appointments, trouble getting transportation, unable to get medications because of transportation. 
Negative examples: no transportation problems, reliable transportation, denies transportation barriers. 
Unclear examples: uses bus, relies on family rides, transportation discussed without saying whether it is a barrier. 
'''

    utilities = '''**utilities**
This domain covers difficulty paying for heating, water, electricity, gas, or risk of shutoff. 
Positive examples: trouble paying utility bills, threatened shutoff, already shut off. 
Negative examples: no utility concerns, utilities are stable and paid. 
Unclear examples: utility assistance discussed without clear need, household bills mentioned without indicating utility risk. 
'''

# Helper Functions

In [5]:
DomainLabel = Literal["positive", "negative", "unclear"]

DOMAIN_NAMES = (
    "child_family_care",
    "education",
    "employment",
    "financial_strain",
    "food_insecurity",
    "housing_instability",
    "housing_quality",
    "relationship_safety",
    "social_isolation",
    "stress",
    "transportation",
    "utilities",
)

VALID_LABELS = {"positive", "negative", "unclear"}


class DomainSpec(TypedDict, total=False):
    label: DomainLabel
    mention: bool


def build_prompt(
    cfg: syn_config,
    domain_specs: Mapping[str, DomainSpec],
    n_single_generate: int = syn_config.n_generate_single,
    n_generate_mixture: int = syn_config.n_generate_mixture,
) -> str:
    """
    Build a synthetic clinical-note generation prompt.

    Parameters
    ----------
    domain_specs
        Example:
        {
            "food_insecurity": {
                "label": "positive",
            },
            "transportation": {
                "label": "negative",
            },
            "housing_instability": {
                "label": "unclear",
                "mention": True,
            },
            "utilities": {
                "label": "unclear",
                "mention": False,
            },
        }
    """

    if not domain_specs:
        raise ValueError("Provide at least one domain specification.")

    invalid_domains = set(domain_specs) - set(DOMAIN_NAMES)
    if invalid_domains:
        raise ValueError(
            f"Unknown domain(s): {sorted(invalid_domains)}. "
            f"Valid domains are: {list(DOMAIN_NAMES)}"
        )

    normalized_specs: dict[str, dict[str, DomainLabel | bool]] = {}

    for domain_name, spec in domain_specs.items():
        if "label" not in spec:
            raise ValueError(f"Missing 'label' for domain: {domain_name}")

        label = spec["label"]
        if label not in VALID_LABELS:
            raise ValueError(
                f"Invalid label '{label}' for {domain_name}. "
                f"Allowed labels: {sorted(VALID_LABELS)}"
            )

        # Positive/negative must always be documented.
        # Unclear may be documented or intentionally absent.
        mention = spec.get("mention", True)

        if label in {"positive", "negative"} and not mention:
            raise ValueError(
                f"{domain_name} is '{label}', so mention must be True. "
                "Positive and negative labels require explicit documentation."
            )

        normalized_specs[domain_name] = {
            "label": label,
            "mention": mention,
        }

    # Domains the generated note must explicitly discuss.
    mentioned_domains = {
        domain_name: spec
        for domain_name, spec in normalized_specs.items()
        if spec["mention"]
    }

    # Domains explicitly configured as unclear but intentionally absent.
    hidden_unclear_domains = {
        domain_name
        for domain_name, spec in normalized_specs.items()
        if spec["label"] == "unclear" and not spec["mention"]
    }

    # All domains not supplied by the caller are also forbidden.
    unspecified_domains = set(DOMAIN_NAMES) - set(domain_specs)

    # No content should discuss, imply, or otherwise signal these domains.
    excluded_domains = hidden_unclear_domains | unspecified_domains

    is_single_domain = len(domain_specs) == 1
    n_generate = (
        n_single_generate
        if is_single_domain
        else n_generate_mixture
    )

    note_lengths = (
        "64 and 128 tokens"
        if is_single_domain
        else "128 and 256 tokens"
    )

    selected_definitions = "\n\n".join(
        f"### {domain_name}\n"
        f"Required label: **{spec['label'].upper()}**\n\n"
        f"{getattr(cfg, domain_name).strip()}"
        for domain_name, spec in mentioned_domains.items()
    )

    label_instructions: list[str] = []

    for domain_name, spec in mentioned_domains.items():
        label = spec["label"]

        if label == "positive":
            instruction = (
                f"- **{domain_name} = POSITIVE, MUST MENTION**"
            )
        elif label == "negative":
            instruction = (
                f"- **{domain_name} = NEGATIVE, MUST MENTION** "
            )
        else:
            instruction = (
                f"- **{domain_name} = UNCLEAR, MUST MENTION**"
            )

        label_instructions.append(instruction)

    for domain_name in sorted(excluded_domains):
        if domain_name in hidden_unclear_domains:
            reason = (
                "This domain was explicitly specified as UNCLEAR with mention=False."
            )
        else:
            reason = (
                "This domain was not included in the requested domain specifications."
            )

        label_instructions.append(
            f"- **{domain_name} = DO NOT COVER** "
        )

    domain_rules_text = "\n".join(label_instructions)

    prohibited_domains_text = (
        ", ".join(sorted(excluded_domains))
        if excluded_domains
        else "None"
    )

    prompt = dedent(
        f"""
Given the domain definitions and required labeling rules below, generate {n_generate} synthetic, deidentified clinical notes. Each note must cover every domain marked MUST MENTION and must not cover any domain marked as DO NOT COVER.

Domain definitions:
**child_family_care**
This domain covers child care, daycare, dependent care, or difficulty paying for child care. 
Positive examples: needs daycare, cannot afford child care, child care is a barrier. 
Negative examples: no child care needs, child care is adequate, dependent care needs are met. 
Unclear examples: has children, child care discussed, family responsibilities mentioned without stating a need. 

**education** 
This domain covers educational attainment or education-related disadvantage. 
Positive examples: less than high school diploma, very limited schooling, education level suggests disadvantage. 
Negative examples: high school graduate, GED, or higher education clearly documented. 
Unclear examples: education discussed without a clear level, school history mentioned without enough detail. 
Important: do not treat any mention of school as positive unless low attainment or education-related disadvantage is clearly documented.  

**employment** 
This domain covers unemployment, job instability, or needing help finding work. 
Positive examples: unemployed and looking for work, wants help finding a job, cannot keep a job. 
Negative examples: employed, stable work, no employment assistance needed. 
Unclear examples: job discussed without clear current status, retired/disability status mentioned without clear employment need. 

**financial_strain**
This domain covers difficulty paying for basics such as food, housing, heating, medical care, medications, or other essential expenses. 
Positive examples: somewhat hard or very hard to pay for basics, trouble paying for medicines, cannot afford bills. 
Negative examples: not hard to pay for basics, denies financial concerns, medications are affordable. 
Unclear examples: money concerns mentioned without indicating whether basic needs are affected. 

**food_insecurity**
This domain covers worrying food will run out, food actually running out, skipping meals, or not having enough money for food. 
Positive examples: often true/sometimes true that food runs out, household ran out of food, cannot afford food. 
Negative examples: food secure, denies food insecurity, enough food available. 
Unclear examples: appetite issues, diet changes, or nutrition counseling without evidence of access problems. 

**housing_instability**
This domain covers unstable housing, homelessness risk, risk of losing housing, eviction risk, or inability to pay rent/mortgage on time. 
Positive examples: worried about losing housing, no steady place to live, unable to pay rent or mortgage on time. 
Negative examples: steady place to live, housing stable, denies housing concerns. 
Unclear examples: housing mentioned without stability status, temporarily staying somewhere but unclear whether this reflects instability. 


**housing_quality**
This domain covers unsafe or poor housing conditions. 
Positive examples: pests, bugs, mice, inadequate heat, mold, leaks, unsafe housing conditions. 
Negative examples: home conditions are adequate, safe, or no housing quality problems. 
Unclear examples: housing discussed without condition details.

**relationship_safety**
This domain covers threats, physical harm, abuse, or feeling unsafe with family, friends, partner, or others. 
Positive examples: threatened with harm, physically hurt, abuse, feels unsafe in relationship/home. 
Negative examples: denies abuse, denies violence, feels safe. 
Unclear examples: safety screening mentioned without result, interpersonal conflict mentioned without clear threat/harm. 

**social_isolation**
This domain covers loneliness, isolation, or very limited meaningful contact with others. 
Positive examples: feels lonely, feels isolated, sees/talks to close people less than once a week. 
Negative examples: regular supportive contact, denies loneliness or isolation. 
Unclear examples: lives alone, few family details, limited social history without clear loneliness or support deficit. 

**stress** 
This domain covers clinically meaningful current stress burden. 
Positive examples: stressed, quite a bit of stress, ongoing stress affecting daily life. 
Negative examples: denies stress, no current stress concerns. 
Unclear examples: anxiety/depression history without explicit current stress, stress discussed in general terms only. 
Important: do not classify a psychiatric diagnosis alone as stress unless the note explicitly describes current stress. 


**transportation**
This domain covers lack of transportation interfering with medical appointments, medications, work, meetings, or daily living needs. 
Positive examples: transportation kept patient from appointments, trouble getting transportation, unable to get medications because of transportation. 
Negative examples: no transportation problems, reliable transportation, denies transportation barriers. 
Unclear examples: uses bus, relies on family rides, transportation discussed without saying whether it is a barrier. 

**utilities**
This domain covers difficulty paying for heating, water, electricity, gas, or risk of shutoff. 
Positive examples: trouble paying utility bills, threatened shutoff, already shut off. 
Negative examples: no utility concerns, utilities are stable and paid. 
Unclear examples: utility assistance discussed without clear need, household bills mentioned without indicating utility risk. 

Required domain behavior:
{domain_rules_text}

Excluded domains:
{prohibited_domains_text}

Generate notes with target lengths of {note_lengths}.
Match realistic deidentified clinical-note from EHR structure, language,
abbreviations.

Return ONLY a valid Python list of strings. The output will be passed to
ast.literal_eval(), so do not include Markdown fences, explanations,
headings, or text outside the Python list.

Requirements:
- Every note must satisfy every listed domain rule and instruction provided.
- Explicitly cover only domains marked POSITIVE, NEGATIVE, or
  (UNCLEAR and MUST MENTION) and other domains should not be covered in the clinical note.
- Do not contradict a domain's required status within a note.
- For POSITIVE domains, include clear evidence.
- For NEGATIVE domains, include clear evidence.
- For UNCLEAR domains marked MUST MENTION, discuss the domain without
  enough evidence to assign positive or negative status.
- For every domain marked DO NOT COVER, do not mention, screen for, deny, imply, refer for, provide resources for, or include indirect
  clues about that domain or indirectly/directly covered in domains labeled as POSITIVE or NEGATIVE or UNCLEAR.
- Do not include standard social-history template language if it could accidentally introduce an excluded domain.
- Do not include patient names, MRNs, addresses, phone numbers, dates of birth, or other directly identifying information.
- Vary wording, note type, clinical context, and severity across examples.
- Do not call the text "synthetic.
- Make sure when only child_family_care, food_insecurity or housing_instability is POSITIVE, don't cover it using difficulty on paying or affordability in any of the generated notes.
"""
    ).strip()

    return prompt

In [ ]:
def run_prompt(prompt_text, model_id = syn_config.model_id):

    llm_model_id = f"arn:aws:bedrock:us-east-1:752168372872:application-inference-profile/{model_id}"
    conversation = [
            {
                "role": "user",
                "content": [{"text": prompt_text}]
            }
        ]
    response = br.converse(
                    modelId=llm_model_id,
                    messages=conversation,
                    inferenceConfig={"maxTokens": 8000, "temperature": 0.5},
                )
    result_text = response["output"]["message"]["content"][0]["text"]

    return result_text


# Generate Data Experiments

In [ ]:
sample_prompt = build_prompt(cfg = syn_config, domain_specs={
            "child_family_care": {
                "label": "positive",
            }
        })

print(sample_prompt)

In [ ]:
sample_notes = run_prompt(sample_prompt)

In [ ]:
print(ast.literal_eval(sample_notes)[1])

# Generate Data

In [ ]:
# DOMAIN_NAMES = (
#     "child_family_care",
#     "education",
#     "employment",
#     "financial_strain",
#     "food_insecurity",
#     "housing_instability",
#     "housing_quality",
#     "relationship_safety",
#     "social_isolation",
#     "stress",
#     "transportation",
#     "utilities",
# )

In [ ]:
# Positive
child_family_care_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "child_family_care": {"label": "positive"},
    },
)))

education_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "education": {"label": "positive"},
    },
)))

employment_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "employment": {"label": "positive"},
    },
)))

financial_strain_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "financial_strain": {"label": "positive"},
    },
)))

food_insecurity_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "food_insecurity": {"label": "positive"},
    },
)))

housing_instability_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "housing_instability": {"label": "positive"},
    },
)))

housing_quality_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "housing_quality": {"label": "positive"},
    },
)))

relationship_safety_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "relationship_safety": {"label": "positive"},
    },
)))

social_isolation_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "social_isolation": {"label": "positive"},
    },
)))

stress_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "stress": {"label": "positive"},
    },
)))

transportation_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "transportation": {"label": "positive"},
    },
)))

utilities_syn_data_pos = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "utilities": {"label": "positive"},
    },
)))

In [ ]:
import pickle
from pathlib import Path

output_dir = Path("synthetic_data")
output_dir.mkdir(parents=True, exist_ok=True)

synthetic_datasets = {
    "child_family_care_syn_data_pos": child_family_care_syn_data_pos,
    "education_syn_data_pos": education_syn_data_pos,
    "employment_syn_data_pos": employment_syn_data_pos,
    "financial_strain_syn_data_pos": financial_strain_syn_data_pos,
    "food_insecurity_syn_data_pos": food_insecurity_syn_data_pos,
    "housing_instability_syn_data_pos": housing_instability_syn_data_pos,
    "housing_quality_syn_data_pos": housing_quality_syn_data_pos,
    "relationship_safety_syn_data_pos": relationship_safety_syn_data_pos,
    "social_isolation_syn_data_pos": social_isolation_syn_data_pos,
    "stress_syn_data_pos": stress_syn_data_pos,
    "transportation_syn_data_pos": transportation_syn_data_pos,
    "utilities_syn_data_pos": utilities_syn_data_pos,
}

for dataset_name, dataset in synthetic_datasets.items():
    output_path = output_dir / f"{dataset_name}.pkl"
    
    with open(output_path, "wb") as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Saved: {output_path}")

## Negative

In [ ]:
# Negative
child_family_care_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "child_family_care": {
            "label": "negative",
        }})))

education_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "education": {
            "label": "negative",
        }})))

employment_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "employment": {
            "label": "negative",
        }})))

financial_strain_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "financial_strain": {
            "label": "negative",
        }})))

food_insecurity_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "food_insecurity": {
            "label": "negative",
        }})))

housing_instability_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "housing_instability": {
            "label": "negative",
        }})))

housing_quality_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "housing_quality": {"label": "negative"},
    },
)))

relationship_safety_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "relationship_safety": {
            "label": "negative",
        }})))



social_isolation_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "social_isolation": {
            "label": "negative",
        }})))

stress_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "stress": {
            "label": "negative",
        }})))

transportation_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "transportation": {
            "label": "negative",
        }})))

utilities_syn_data_neg = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "utilities": {
            "label": "negative",
        }})))

In [ ]:
import pickle
from pathlib import Path

output_dir = Path("synthetic_data")
output_dir.mkdir(parents=True, exist_ok=True)

synthetic_datasets = {
    "child_family_care_syn_data_neg": child_family_care_syn_data_neg,
    "education_syn_data_neg": education_syn_data_neg,
    "employment_syn_data_neg": employment_syn_data_neg,
    "financial_strain_syn_data_neg": financial_strain_syn_data_neg,
    "food_insecurity_syn_data_neg": food_insecurity_syn_data_neg,
    "housing_instability_syn_data_neg": housing_instability_syn_data_neg,
    "housing_quality_syn_data_neg": housing_quality_syn_data_neg,
    "relationship_safety_syn_data_neg": relationship_safety_syn_data_neg,
    "social_isolation_syn_data_neg": social_isolation_syn_data_neg,
    "stress_syn_data_neg": stress_syn_data_neg,
    "transportation_syn_data_neg": transportation_syn_data_neg,
    "utilities_syn_data_neg": utilities_syn_data_neg,
}

for dataset_name, dataset in synthetic_datasets.items():
    output_path = output_dir / f"{dataset_name}.pkl"
    
    with open(output_path, "wb") as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Saved: {output_path}")

In [ ]:
child_family_care_syn_data_neg[0]

## Unclear

In [ ]:
# Unclear
child_family_care_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "child_family_care": {
            "label": "unclear",
            "mention": True,
        }})))

education_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "education": {
            "label": "unclear",
            "mention": True,
        }})))

employment_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "employment": {
            "label": "unclear",
            "mention": True,
        }})))

financial_strain_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "financial_strain": {
            "label": "unclear",
            "mention": True,
        }})))

food_insecurity_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "food_insecurity": {
            "label": "unclear",
            "mention": True,
        }})))

housing_instability_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "housing_instability": {
            "label": "unclear",
            "mention": True,
        }})))

housing_quality_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(
    cfg=syn_config,
    domain_specs={
        "housing_quality": {
            "label": "unclear",
            "mention": True,
        }})))

relationship_safety_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "relationship_safety": {
            "label": "unclear",
            "mention": True,
        }})))

social_isolation_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "social_isolation": {
            "label": "unclear",
            "mention": True,
        }})))

stress_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "stress": {
            "label": "unclear",
            "mention": True,
        }})))

transportation_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "transportation": {
            "label": "unclear",
            "mention": True,
        }})))

utilities_syn_data_unclear = ast.literal_eval(run_prompt(build_prompt(cfg = syn_config, domain_specs={
        "utilities": {
            "label": "unclear",
            "mention": True,
        }})))

In [ ]:
import pickle
from pathlib import Path

output_dir = Path("synthetic_data")
output_dir.mkdir(parents=True, exist_ok=True)

synthetic_datasets = {
    "child_family_care_syn_data_unclear": child_family_care_syn_data_unclear,
    "education_syn_data_unclear": education_syn_data_unclear,
    "employment_syn_data_unclear": employment_syn_data_unclear,
    "financial_strain_syn_data_unclear": financial_strain_syn_data_unclear,
    "food_insecurity_syn_data_unclear": food_insecurity_syn_data_unclear,
    "housing_instability_syn_data_unclear": housing_instability_syn_data_unclear,
    "housing_quality_syn_data_unclear": housing_quality_syn_data_unclear,
    "relationship_safety_syn_data_unclear": relationship_safety_syn_data_unclear,
    "social_isolation_syn_data_unclear": social_isolation_syn_data_unclear,
    "stress_syn_data_unclear": stress_syn_data_unclear,
    "transportation_syn_data_unclear": transportation_syn_data_unclear,
    "utilities_syn_data_unclear": utilities_syn_data_unclear,
}

for dataset_name, dataset in synthetic_datasets.items():
    output_path = output_dir / f"{dataset_name}.pkl"
    
    with open(output_path, "wb") as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Saved: {output_path}")

## Mixed

In [ ]:
label_mapping = {
    "positive": 0,
    "negative": 1,
    "unclear": 2,
    "none": 3,
}

domain_order = [
    "child_family_care",
    "education",
    "employment",
    "financial_strain",
    "food_insecurity",
    "housing_instability",
    "housing_quality",
    "relationship_safety",
    "social_isolation",
    "stress",
    "transportation",
    "utilities",
]

In [ ]:
# mixed_1
req_